# Fine-tuning 전후 비교 평가

`2. QLoRA Fine-tuning.ipynb`에서 학습한 어댑터를 base 모델과 **정량적으로 비교**합니다. fine-tuning 전후를 같은 평가셋·같은 프롬프트로 비교하는 과정을 실습합니다.


In [ ]:
import torch
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration
from peft import PeftModel
from qwen_vl_utils import process_vision_info

import vlm_finetune_utils as vu

MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"
ADAPTER_DIR = "outputs/qlora-defect-adapter"

val_records = vu.load_jsonl("data/val/annotations.jsonl")


## 1. base 모델과 (base + QLoRA 어댑터) 모델 로드

In [ ]:
processor = AutoProcessor.from_pretrained(MODEL_ID)

base_model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto"
)

tuned_model = PeftModel.from_pretrained(
    Qwen2VLForConditionalGeneration.from_pretrained(
        MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto"
    ),
    ADAPTER_DIR,
)


## 2. 추론 함수

In [ ]:
def ask(model, record, image_root):
    messages = vu.to_chat_messages(record, image_root)[:1]  # human 턴만 전달
    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, _ = process_vision_info(messages)
    inputs = processor(
        text=[text], images=image_inputs, return_tensors="pt", padding=True
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=64)
    trimmed = output_ids[:, inputs["input_ids"].shape[1]:]
    return processor.batch_decode(trimmed, skip_special_tokens=True)[0].strip()


## 3. 검증셋 전체에 대해 두 모델 비교

In [ ]:
rows = []
for r in val_records:
    base_answer = ask(base_model, r, "data/val")
    tuned_answer = ask(tuned_model, r, "data/val")
    rows.append(
        {
            "label": r["label"],
            "base_answer": base_answer,
            "base_correct": vu.judge_answer(base_answer, r["label"]),
            "tuned_answer": tuned_answer,
            "tuned_correct": vu.judge_answer(tuned_answer, r["label"]),
        }
    )

for row in rows[:5]:
    print(row)


In [ ]:
base_acc = sum(r["base_correct"] for r in rows) / len(rows)
tuned_acc = sum(r["tuned_correct"] for r in rows) / len(rows)

print(f"base 모델 정확도:  {base_acc:.0%}")
print(f"QLoRA 모델 정확도: {tuned_acc:.0%}")


## 결과 해석
- base 모델은 "불량 유무"는 맞혀도 "어느 모서리인지"까지는 형식이 흔들리는 경우가 많습니다.
- QLoRA 모델은 학습 데이터의 응답 포맷("OO 모서리에 스크래치가 있습니다")을 그대로 재현하는 경향이 강해져 `judge_answer`의 키워드 매칭 정확도가 올라가는 것을 확인할 수 있습니다.
- 실무에서는 이런 단순 키워드 매칭 대신, 실제 라벨과의 의미적 일치 여부를 사람이 검수하거나 별도 평가 모델을 활용하는 것이 좋습니다.